# PhysIR — Stage 3: Blur Arm Probe (Kaggle)

**Question:** does a *low-weight* blur term contribute once phase carries the load?

This is **not** a repeat of what already failed. Different corner of the space:

| run | blur | phase | result |
|---|---|---|---|
| Stage 2 combined | **0.17** (high) | **0.02** (low) | 24.34 — blur conflicted |
| noedge probe | 0.05 | — | blur vs edge conflict |
| **blur arm (this)** | **0.02** (low) | **0.20** (high) | ? |

**Controlled comparison:** identical to `sweep_ph20_re18` in every way except `blur=0.02`.

| baseline | PSNR | SSIM | LPIPS |
|---|---|---|---|
| `sweep_ph20_re18` (blur off) | 25.1927 | 0.8399 | 0.2719 |

### Decision rule (important)

The Stage 3 extension exposed a **~0.13 dB PSNR noise floor** at 30 ep. So:

> **Positive only if SSIM improves AND LPIPS improves.**
> PSNR-only movement, or mixed signals → noise → treat as negative.

- **Negative/flat (expected):** Stage 4 = `ph20_re18`, blur OFF. Nothing changes.
- **Positive:** Stage 4 adds `blur: 0.02` (and warrants a quick blur-weight mini-probe).

---

> ⚠️ **This run has `blur: True` → a learnable PSF (σ).** The resume σ-reset bug applies.
> **The `physir-code` Kaggle dataset MUST contain the fixed `train.py`** (with
> `restore_phys_state`). Otherwise an interrupted session silently resets σ and the
> comparison is invalid. On a clean resume you'll see `Restored physics-loss state (blur σ=…)`.

**Hands-off:** idempotent + resume-safe — re-run the notebook if the session dies.
**Env:** code + data are Kaggle input datasets (read-only, no clone); cwd stays
`/kaggle/working`.  **Accelerator:** GPU T4 × 2.  ~3.5–4 hr.

## 1. Install dependencies

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths — edit before running

In [ ]:
import os

# ── EDIT THESE (Kaggle input datasets — read-only) ────────────────────────────
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'      # MUST have fixed train.py
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
CHECKPOINT_DIR = '/kaggle/working'
# ──────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working'   # cwd stays here → .metrics writes to /kaggle/working/.metrics
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
assert os.path.isdir(CODE_DIR), f'CODE_DIR not found: {CODE_DIR}'
assert os.path.isfile(os.path.join(CODE_DIR, 'train.py')), 'train.py not in CODE_DIR'

# Guard: confirm the resume fix is present (blur has a learnable σ).
_src = open(os.path.join(CODE_DIR, 'train.py')).read()
if 'restore_phys_state' in _src and 'phys_state' in _src:
    print('✅ train.py has the phys-loss resume fix (restore_phys_state).')
else:
    print('⚠️  WARNING: train.py is MISSING the resume fix. If this session is interrupted,')
    print('    the blur PSF (σ) will silently reset and invalidate the comparison.')
    print('    Upload a new version of the physir-code dataset before running.')

print(f'\nCode:        {CODE_DIR}')
print(f'Dataset:     {LOLBLUR_ROOT}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## 3. Train/val split (same 90/10, seed 42 — identical to the sweep)

In [ ]:
import random
from pathlib import Path

SUBS       = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC  = Path(LOLBLUR_ROOT) / 'train'
SPLIT_ROOT = Path(WORKING_DIR) / 'LOLBlur_split'

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips   = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'Clips — train: {len(train_clips)}, val: {len(val_clips)}')

for split_name, clip_list in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        dst_base = SPLIT_ROOT / split_name / sub
        dst_base.mkdir(parents=True, exist_ok=True)
        for clip in clip_list:
            src_path = (TRAIN_SRC / sub / clip).resolve()
            dst_path = dst_base / clip
            if not dst_path.exists():
                dst_path.symlink_to(src_path)

PHYSIR_TRAIN = str(SPLIT_ROOT / 'physir_train')
PHYSIR_VAL   = str(SPLIT_ROOT / 'physir_val')
print(f'  train: {PHYSIR_TRAIN}')
print(f'  val:   {PHYSIR_VAL}')

## 4. Blur arm — config + hands-off run

Idempotent: skips if already at 30 ep, resumes if partial. Re-run freely.

In [ ]:
import yaml, csv, subprocess
from pathlib import Path

EPOCHS = 30

# ── Blur arm: Stage 4 candidate weights + a low-weight blur term ──────────────
# Base is the recommended Stage 4 config. Switch to phase 0.12 if you prefer the
# max-PSNR candidate (ph12_re18) — but then compare against ITS numbers, not ph20's.
BASE = {'phase': 0.20, 'retinex': 0.18}
BLUR_WEIGHT = 0.02
RUN_NAME = 'blur_arm_ph20_re18'

# Reference: the identical run WITHOUT blur (sweep_ph20_re18)
REF = {'name': 'sweep_ph20_re18', 'psnr': 25.1927, 'ssim': 0.8399, 'lpips': 0.2719}
# ──────────────────────────────────────────────────────────────────────────────

cfg = {
    'datasets': {
        'name': 'LOLBlur',
        'train': {'low_path': f'{PHYSIR_TRAIN}/low_blur_noise',
                  'high_path': f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size': 256, 'batch_size': 4, 'n_workers': 4, 'use_flips': True},
        'val': {'test_path': PHYSIR_VAL, 'batch_size_test': 1,
                'verbose': False, 'n_workers': 4},
    },
    'network': {'name': 'DarkIR', 'img_channels': 3, 'width': 32,
                'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
                'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
                'dilations': [1, 4, 9], 'extra_depth_wise': True},
    'save': {'pretrained': None,
             'path': f'{CHECKPOINT_DIR}/{RUN_NAME}_latest.pt',
             'best': f'{CHECKPOINT_DIR}/{RUN_NAME}_best.pt',
             'save_freq': 5, 'eval_freq': 5, 'perceptual_eval_epochs': []},
    'pixel_criterion': 'l1', 'perceptual': False,
    'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
    'frequency': False,
    'enhance': True, 'enhance_weight': 0.01,
    'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
    'physics_losses': {
        'enabled': True,
        'phase2_start': 9999, 'phase3_start': 9999, 'phase4_start': 1,  # all gates from ep1
        'retinex': True, 'retinex_weight': BASE['retinex'],
        'blur':    True, 'blur_weight':    BLUR_WEIGHT,   # ← the variable under test
        'phase':   True, 'phase_weight':   BASE['phase'],
    },
    'train': {'epochs': EPOCHS, 'lr_initial': 0.00028, 'weight_decay': 0.001,
              'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0},
    'wandb': {'init': False, 'name': RUN_NAME},
    'Resize': False,
}

cfg_path = f'{WORKING_DIR}/{RUN_NAME}.yml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

def done_epoch(run_name):
    log = Path(WORKING_DIR) / '.metrics' / run_name / 'training_log.csv'
    if not log.exists():
        return 0
    last = 0
    try:
        with open(log) as f:
            for r in csv.DictReader(f):
                last = max(last, int(r['epoch']))
    except Exception:
        pass
    return last

pl = cfg['physics_losses']
print(f'{RUN_NAME}:  phase={pl["phase_weight"]}  retinex={pl["retinex_weight"]}  '
      f'blur={pl["blur_weight"]}')
print(f'vs reference {REF["name"]}: PSNR={REF["psnr"]}  SSIM={REF["ssim"]}  LPIPS={REF["lpips"]}\n')

d = done_epoch(RUN_NAME)
if d >= EPOCHS:
    print(f'✅ {RUN_NAME} already at {d}/{EPOCHS} ep — skipping. Run the results cell.')
else:
    tag = f'(resuming from ep {d})' if d > 0 else '(from scratch)'
    print(f'===== Running {RUN_NAME} {tag} =====')
    r = subprocess.run(['torchrun', '--nproc_per_node=2',
                        f'{CODE_DIR}/train.py', '-p', cfg_path])
    print(f'===== finished (exit={r.returncode}) =====')

## 5. Verdict — apply the SSIM-AND-LPIPS decision rule

In [ ]:
import pandas as pd
from pathlib import Path

log = Path(WORKING_DIR) / '.metrics' / RUN_NAME / 'training_log.csv'
if not log.exists():
    print('No log yet — training may still be running.')
else:
    df = pd.read_csv(log)
    df.replace('', float('nan'), inplace=True)
    val = df.dropna(subset=['val_psnr'])
    best = val.loc[val['val_psnr'].idxmax()]
    p, s, l = float(best['val_psnr']), float(best['val_ssim']), float(best['val_lpips'])

    print(f'=== Blur arm ({RUN_NAME}) @ {int(df["epoch"].max())} ep ===\n')
    print(f'{"metric":8s} {"blur arm":>10s} {"ref (no blur)":>14s} {"delta":>10s}')
    print('-' * 46)
    print(f'{"PSNR":8s} {p:10.4f} {REF["psnr"]:14.4f} {p-REF["psnr"]:+10.4f}')
    print(f'{"SSIM":8s} {s:10.4f} {REF["ssim"]:14.4f} {s-REF["ssim"]:+10.4f}')
    print(f'{"LPIPS":8s} {l:10.4f} {REF["lpips"]:14.4f} {l-REF["lpips"]:+10.4f}  (lower=better)')

    ssim_better  = s > REF['ssim']
    lpips_better = l < REF['lpips']
    print('\n--- Verdict (PSNR noise floor at 30 ep ≈ 0.13 dB — do not judge on PSNR alone) ---')
    if ssim_better and lpips_better:
        print('  ✅ POSITIVE — SSIM and LPIPS both improve.')
        print('     → Stage 4 should ADD blur=0.02. Consider a blur-weight mini-probe (0.01/0.02/0.05).')
    else:
        print('  ❌ NEGATIVE / FLAT — blur does not contribute (SSIM and LPIPS do not both improve).')
        print('     → Stage 4 = ph20_re18, blur OFF. Blur becomes an ablation row documenting redundancy.')

    # σ trajectory + resume-discontinuity check (blur has a learnable PSF)
    if 'blur_sigma' in df.columns and df['blur_sigma'].notna().any():
        sig = df['blur_sigma'].dropna().astype(float).reset_index(drop=True)
        print(f'\n  blur σ: {sig.iloc[0]:.3f} → {sig.iloc[-1]:.3f}')
        jumps = sig.diff().abs()
        big = jumps[jumps > 0.5]
        if len(big):
            print(f'  ⚠️  σ DISCONTINUITY at epoch(s) {[int(i)+1 for i in big.index]} '
                  f'(max jump {big.max():.3f}) — resume reset the PSF.')
            print('     The comparison is COMPROMISED. Ensure train.py has the fix, then re-run clean.')
        elif sig.iloc[-1] >= 4.9:
            print('  ⚠️  σ pinned near the 5.0 clamp — PSF estimate degenerate. Note for report.')
        else:
            print('  ✅ σ trajectory smooth, no resume reset.')